# nSTATPaperExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `decoding_1d`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/nSTATPaperExamples.md`


Notebook source link: [nSTATPaperExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/nSTATPaperExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "nSTATPaperExamples"
FAMILY = "decoding_1d"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"nSTATPaperExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"nSTATPaperExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"nSTATPaperExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"nSTATPaperExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# nSTATPaperExamples: multi-section paper-style workflow summary.
from nstat.compat.matlab import Analysis, Covariate, CovColl, DecodingAlgorithms, Trial, TrialConfig, nspikeTrain, nstColl

# Section 1: constant-baseline point-process fit (mEPSC-style).
dt = 0.001
time = np.arange(0.0, 8.0, dt)
baseline_rate = 12.0
spike_prob = np.clip(baseline_rate * dt, 0.0, 0.5)
spike_times_const = time[rng.random(time.size) < spike_prob]

baseline_cov = Covariate(time=time, data=np.ones(time.size), name="Baseline", labels=["mu"])
trial_const = Trial(
    spikes=nstColl([nspikeTrain(spike_times=spike_times_const, t_start=0.0, t_end=float(time[-1]), name="epsc")]),
    covariates=CovColl([baseline_cov]),
)
cfg_const = TrialConfig(covariateLabels=["mu"], Fs=1.0 / dt, fitType="poisson", name="Constant Baseline")
fit_const = Analysis.fitTrial(trial_const, cfg_const, unitIndex=0)
lam_const = fit_const.predict(np.ones((time.size, 1)))

# Section 2: explicit-stimulus logistic fit.
stim = np.sin(2.0 * np.pi * 2.0 * time)
eta = -3.1 + 1.2 * stim
p_spk = 1.0 / (1.0 + np.exp(-eta))
y_bin = rng.binomial(1, p_spk)
fit_stim = Analysis.fitGLM(X=stim[:, None], y=y_bin, fitType="binomial", dt=1.0)
p_hat = fit_stim.predict(stim[:, None])

# Section 3: trial-difference matrix and significance markers.
n_trials = 20
trial_mat = np.zeros((n_trials, time.size), dtype=float)
for k in range(n_trials):
    gain = 0.8 + 0.4 * rng.random()
    pk = np.clip((baseline_rate + 6.0 * (stim > 0.25)) * gain * dt, 0.0, 0.8)
    trial_mat[k] = rng.binomial(1, pk)
rate_ci, prob_mat, sig_mat = DecodingAlgorithms.computeSpikeRateCIs(trial_mat)

fig = plt.figure(figsize=(12.0, 9.2))
ax1 = fig.add_subplot(2, 2, 1)
ax1.vlines(spike_times_const, 0.0, 1.0, linewidth=0.4)
ax1.set_title("Paper Exp 1: Constant Mg raster")
ax1.set_xlabel("time [s]")
ax1.set_yticks([])

ax2 = fig.add_subplot(2, 2, 2)
ax2.plot(time, baseline_rate * np.ones_like(time), "k", linewidth=1.1, label="true")
ax2.plot(time, lam_const, "tab:blue", linewidth=1.0, label="fit")
ax2.set_title("Constant-rate fit")
ax2.set_xlabel("time [s]")
ax2.set_ylabel("Hz")
ax2.legend(loc="upper right")

ax3 = fig.add_subplot(2, 2, 3)
ax3.plot(time, p_spk, "k", linewidth=1.1, label="true p(spike)")
ax3.plot(time, p_hat, "tab:red", linewidth=1.0, label="GLM fit")
ax3.set_title("Paper Exp 5: stimulus decoding setup")
ax3.set_xlabel("time [s]")
ax3.set_ylabel("probability")
ax3.legend(loc="upper right")

ax4 = fig.add_subplot(2, 2, 4)
im = ax4.imshow(prob_mat, origin="lower", cmap="gray_r", aspect="auto")
yy, xx = np.where(sig_mat > 0)
if xx.size:
    ax4.plot(xx, yy, "r*", markersize=4)
ax4.set_title("Paper Exp 4: trial significance matrix")
ax4.set_xlabel("trial")
ax4.set_ylabel("trial")
fig.colorbar(im, ax=ax4, fraction=0.04, pad=0.02)
plt.tight_layout()
plt.show()

learning_trial = int(np.argmax(np.any(sig_mat > 0, axis=0)) + 1) if np.any(sig_mat > 0) else 0
assert rate_ci.size > 0
assert prob_mat.shape[0] == n_trials

CHECKPOINT_METRICS = {
    "const_spike_count": float(spike_times_const.size),
    "stim_fit_rmse": float(np.sqrt(np.mean((p_hat - p_spk) ** 2))),
    "learning_trial_index": float(learning_trial),
}
CHECKPOINT_LIMITS = {
    "const_spike_count": (5.0, 5000.0),
    "stim_fit_rmse": (0.0, 0.4),
    "learning_trial_index": (0.0, float(n_trials)),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
